# This code will filtrate and do some preprocessing of the TMT data coming from the "FGCZ"

This code is function oriented but I will try to make the transition to Object Oriented Programming OOP)
The functions this notebook does are:
- Extract the FC information from the dataframe that holds all the information.
- Add the SD and number of replicates per phosphosite
- Filter by Log2FC, number of replicates, and wheter the phosphorylation is placed or not
- Expressing the time course behaviour as a qualitative steps
    

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Extracting information from the "All" dataset, reorganizing and refeactoring

Taking data from the DE_Groups_vs_controls.xlsx sheet: diff_ex_analysis_wide

1. First I am only taking the FC information. 

In [68]:
all_raw_df = pd.read_excel("") # 'Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx', sheet_name='diff_exp_analysis_wide')

keep_columns = ['protein_Id', 'description', 'site', 'diff.full_vs_starve',
                'diff.EGF1_vs_starve', 'diff.EGF2_vs_starve', 'diff.EGF5_vs_starve', 'diff.EGF10_vs_starve', 'diff.EGF90_vs_starve',
                'diff.INS1_vs_starve', 'diff.INS2_vs_starve', 'diff.INS5_vs_starve', 'diff.INS10_vs_starve', 'diff.INS90_vs_starve',
                'diff.EGFnINS1_vs_starve', 'diff.EGFnINS2_vs_starve', 'diff.EGFnINS5_vs_starve', 'diff.EGFnINS10_vs_starve', 'diff.EGFnINS90_vs_starve' ]

clean_df = all_raw_df[keep_columns].copy()

clean_df.rename(columns={'protein_Id' : 'protein_ID',
                         'description' : 'prot_name',
                         'diff.EGF1_vs_starve' : 'EGF_1',
                         'diff.EGF2_vs_starve' : 'EGF_2',
                         'diff.EGF5_vs_starve' : 'EGF_5',
                         'diff.EGF10_vs_starve' : 'EGF_10',
                         'diff.EGF90_vs_starve' : 'EGF_90',
                         'diff.INS1_vs_starve' : 'INS_1',
                         'diff.INS2_vs_starve' : 'INS_2',
                         'diff.INS5_vs_starve' : 'INS_5',
                         'diff.INS10_vs_starve' : 'INS_10',
                         'diff.INS90_vs_starve' : 'INS_90',
                         'diff.EGFnINS1_vs_starve' : 'EGFnINS_1',
                         'diff.EGFnINS2_vs_starve' : 'EGFnINS_2',
                         'diff.EGFnINS5_vs_starve' : 'EGFnINS_5',
                         'diff.EGFnINS10_vs_starve' : 'EGFnINS_10',
                         'diff.EGFnINS90_vs_starve' : 'EGFnINS_90',
                         'diff.full_vs_starve' : 'EGF_full' #EGF_full
                       },inplace=True)

clean_df.insert(4,'EGF_starve',0) 
clean_df.insert(10,'INS_full', clean_df.EGF_full)
clean_df.insert(11,'INS_starve',0)   
clean_df.insert(17,'EGFnINS_full', clean_df.EGF_full) 
clean_df.insert(18,'EGFnINS_starve',0)   

# To this point the dataframe only has the FC information, and the conditions are in continuous way
# clean_df.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC.xlsx", index = False)


# Put the data grouped by conditions
# grouped_columns = ['protein_ID', 'prot_name', 'site', 'condition', 'full', 'starve', 'min1', 'min2', 'min5', 'min10', 'min90']
# grouped_df = pd.DataFrame(columns = grouped_columns)
# 
# for row in clean_df.itertuples():
#     for i in [0,5,10]:
#         if i == 0:
#             condition = 'EGF'
#         elif i == 5:
#             condition = 'INS'
#         elif i == 10:
#             condition = 'EGFnINS'
# 
#         new_row = {'protein_ID' : row[1],
#                    'prot_name' : row[2],
#                    'site' : row[3],
#                    'condition' : condition,
#                    'full' : row[4],
#                    'starve' : row[5],
#                    'min1' : row[6+i],
#                    'min2' : row[7+i],
#                    'min5' : row[8+i],
#                    'min10' : row[9+i],
#                    'min90' : row[10+i]}
# 
#         grouped_df = grouped_df._append(new_row, ignore_index=True)
# 
# grouped_df.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_grouped_FC.xlsx", index = False)


FileNotFoundError: [Errno 2] No such file or directory: ''

## Extract normalized data of abundances. 
I have seen that Normalized abundances has a lot of information that can be missed. So lets see how is this data.

**be careful with protein_ID Q6ZSR9, it has no  prot_name in the database and has to be added manually, it is FLJ45252** 

Keys with the same values (these are the same gene with different entries in uniprot because of different isoforms)

Value 'MACF1' is assigned to keys: ['O94854', 'Q9UPN3']

Value 'RAB34' is assigned to keys: ['P0DI83', 'Q9BZG1']

Value 'ERCC6' is assigned to keys: ['P0DP91', 'Q03468']

Value 'CUX1' is assigned to keys: ['P39880', 'Q13948']

Value 'TMPO' is assigned to keys: ['P42166', 'P42167']

In [42]:
path = 'Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx'
excel_sheet = 'stats_normalized_wide'
abundances_raw_df = pd.read_excel(path, sheet_name=excel_sheet) 

keep_columns = ["protein_Id", "site", 
                "mean_full", "mean_starve", 
                "mean_EGF1", "mean_EGF2", "mean_EGF5", "mean_EGF10", "mean_EGF90", 
                "mean_INS1", "mean_INS2", "mean_INS5", "mean_INS10", "mean_INS90", 
                "mean_EGFnINS1", "mean_EGFnINS2", "mean_EGFnINS5", "mean_EGFnINS10", "mean_EGFnINS90",
                "not_na_starve",
                "sd_All",
                "sd_full", "sd_starve", 
                "sd_EGF1", "sd_EGF2", 'sd_EGF5', "sd_EGF10", "sd_EGF90", 
                "sd_INS1", "sd_INS2", "sd_INS5", "sd_INS10", "sd_INS90", 
                "sd_EGFnINS1", "sd_EGFnINS2", "sd_EGFnINS5", "sd_EGFnINS10", "sd_EGFnINS90"]
normalized_abundances_df = abundances_raw_df[keep_columns].copy()
normalized_abundances_df.rename(columns={'protein_Id' : 'protein_ID',
                                         'mean_full' : 'mean_EGF_full',
                                         'mean_starve' : 'mean_EGF_starve',
                                         'not_na_starve' : 'n_replicates',
                                         'sd_full' : 'sd_EGF_full',
                                         'sd_starve': 'sd_EGF_starve'}, inplace=True)

df_with_protnames = pd.read_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC.xlsx")
names_dict = dict(zip(df_with_protnames['protein_ID'], df_with_protnames['prot_name']))

normalized_abundances_df.insert(1,'prot_name', 0)
for row in normalized_abundances_df.itertuples():
    index = row[0]
    id = row[1]
    normalized_abundances_df.at[index, 'prot_name'] = names_dict[id]
normalized_abundances_df.insert(10, 'mean_INS_full', normalized_abundances_df.mean_EGF_full )
normalized_abundances_df.insert(11, 'mean_INS_starve', normalized_abundances_df.mean_EGF_starve )
normalized_abundances_df.insert(17, 'mean_EGFnINS_full', normalized_abundances_df.mean_EGF_full )
normalized_abundances_df.insert(18, 'mean_EGFnINS_starve', normalized_abundances_df.mean_EGF_starve )

normalized_abundances_df.insert(33, 'sd_INS_full', normalized_abundances_df.sd_EGF_full )
normalized_abundances_df.insert(34, 'sd_INS_starve', normalized_abundances_df.sd_EGF_starve )
normalized_abundances_df.insert(40, 'sd_EGFnINS_full', normalized_abundances_df.sd_EGF_full )
normalized_abundances_df.insert(41, 'sd_EGFnINS_starve', normalized_abundances_df.sd_EGF_starve )

# normalized_abundances_df.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_normalized_nrep_SD.xlsx", index = False)
normalized_abundances_df

/var/folders/3x/34lr70ks7312krzcy07d2900gmckg0/T/ipykernel_98914/919515587.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'HNRNPA1L3' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  normalized_abundances_df.at[index, 'prot_name'] = names_dict[id]


,protein_ID,prot_name,site,mean_EGF_full,mean_EGF_starve,mean_EGF1,mean_EGF2,mean_EGF5,mean_EGF10,mean_EGF90,...,sd_INS5,sd_INS10,sd_INS90,sd_EGFnINS_full,sd_EGFnINS_starve,sd_EGFnINS1,sd_EGFnINS2,sd_EGFnINS5,sd_EGFnINS10,sd_EGFnINS90
0,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,14.807128,13.465341,13.419558,13.408277,13.283449,13.761856,13.618333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,21.343344,18.592727,18.517095,18.644140,19.146568,19.664037,19.207402,...,0.132813,0.082451,0.158647,0.240242,0.073607,0.213038,0.017743,0.137722,0.158750,0.203929
2,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,17.775360,18.251348,18.196227,18.079029,18.044772,18.070157,17.793066,...,0.122319,0.050334,0.136001,0.110507,0.109621,0.049720,0.281925,0.100414,0.150921,0.137177
3,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,14.827755,15.165562,15.265463,15.192356,15.134592,15.093879,15.104278,...,0.229781,0.228683,0.101860,0.015341,0.096143,0.177090,0.027352,0.391594,0.279341,0.065133
4,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,18.515057,19.241331,19.137821,18.852081,18.674344,18.720865,18.491244,...,0.096032,0.092438,0.135065,0.190606,0.107901,0.087274,0.362086,0.085434,0.223696,0.084909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_338_1_1_S338~SLsFEMQQDELIEK,14.808259,14.777620,14.885093,14.768630,14.838516,14.793259,14.782368,...,0.071151,0.118530,0.209883,0.140928,0.190284,0.162495,0.153906,0.077442,0.173945,0.194352
34998,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,14.165458,14.491413,14.397329,14.341868,14.388180,14.386649,14.355432,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34999,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_1_S338~SLsFEMQQDELIEKPMSPMQYAR,17.555270,17.465450,17.468174,17.547552,17.490696,17.480868,17.576683,...,0.072786,0.092533,0.092859,0.181606,0.183772,0.186855,0.092704,0.159401,0.079835,0.147529
35000,Q9Y6Y8,SEC23IP,Q9Y6Y8_118_138_1_0~PLTALPFTTGSQDVSNAFSPSISK,12.026680,12.100700,11.984762,12.021384,12.305367,12.241909,12.412716,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Add the standard deviation (SD), the number of replicates

For the SD i am going to take the values from [Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx](Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx) the sheet **stats_normalized_wide**

From here I am onl taking the "sd", however I could also select the "var" and "mean"



In [67]:
path_to_add = "" # "Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC.xlsx"
path_to_stats = "Experiment/1_hTERT_HME1/Data/All/DE_Groups_vs_Controls.xlsx"
path_to_save = path_to_add.replace('.xlsx','_nrep_SD.xlsx')

df_to_add = pd.read_excel(path_to_add)
# df_to_add = clean_df.copy()
stats_df = pd.read_excel(path_to_stats, sheet_name='stats_normalized_wide')

keep_stats2 = ['protein_Id', 'site', 'not_na_full', 'sd_All',
              'sd_full_EGF', 'sd_starve_EGF', 'sd_EGF1', 'sd_EGF2', 'sd_EGF5', 'sd_EGF10', 'sd_EGF90',
              'sd_full_INS', 'sd_starve_INS', 'sd_INS1', 'sd_INS2', 'sd_INS5',  'sd_INS10', 'sd_INS90',
              'sd_full_EGFnINS', 'sd_starve_EGFnINS', 'sd_EGFnINS1', 'sd_EGFnINS2', 'sd_EGFnINS5', 'sd_EGFnINS10', 'sd_EGFnINS90']

df_to_add[keep_stats2[2:]] = 0
df_to_add.rename(columns={'not_na_full' : 'n_replicates'},inplace=True)

for row in df_to_add.itertuples(): # 47 minutes to run (around 2 mins for each condition)
    id = row[3]
    df_to_add.n_replicates.loc[df_to_add['site'] == id ] = stats_df.not_na_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_All.loc[df_to_add['site'] == id ] = stats_df.sd_All.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_EGF.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_EGF.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF1.loc[df_to_add['site'] == id ] = stats_df.sd_EGF1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF2.loc[df_to_add['site'] == id ] = stats_df.sd_EGF2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF5.loc[df_to_add['site'] == id ] = stats_df.sd_EGF5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF10.loc[df_to_add['site'] == id ] = stats_df.sd_EGF10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGF90.loc[df_to_add['site'] == id ] = stats_df.sd_EGF90.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_INS.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_INS.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS1.loc[df_to_add['site'] == id ] = stats_df.sd_INS1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS2.loc[df_to_add['site'] == id ] = stats_df.sd_INS2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS5.loc[df_to_add['site'] == id ] = stats_df.sd_INS5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS10.loc[df_to_add['site'] == id ] = stats_df.sd_INS10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_INS90.loc[df_to_add['site'] == id ] = stats_df.sd_INS90.loc[stats_df['site'] == id].iloc[0]

    df_to_add.sd_full_EGFnINS.loc[df_to_add['site'] == id ] = stats_df.sd_full.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_starve_EGFnINS.loc[df_to_add['site'] == id ] = stats_df.sd_starve.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS1.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS1.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS2.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS2.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS5.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS5.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS10.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS10.loc[stats_df['site'] == id].iloc[0]
    df_to_add.sd_EGFnINS90.loc[df_to_add['site'] == id ] = stats_df.sd_EGFnINS90.loc[stats_df['site'] == id].iloc[0]

df_to_add.to_excel("Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx", index = False)

FileNotFoundError: [Errno 2] No such file or directory: ''

## Filter by Log2FC values, number of replicates and/or number of phosphorylations identified in the peptide

In [2]:
def check_max_amplitude(row):
    '''Returnd the maximun amplitude of the phosphosite'''
    c = 0
    for element in row[:24]:
        if type(element) == float:
            FC_list = row[c:24]
            max_FC = max(FC_list)
            break
        c += 1
    return max_FC

def check_min_amplitude(row):
    '''Returnd the maximun amplitude of the phosphosite'''
    c = 0
    for element in row[:24]:
        if type(element) == float:
            FC_list = row[c:24]
            min_FC = min(FC_list)
            break
        c += 1
    return min_FC

def check_all_localized(phosphorylation_site):
    '''Returns true or false deppending if all phosphorylations sites are localized '''
    phosphopeptide_list = re.split("_|~", phosphorylation_site)
    n_ph = phosphopeptide_list[3]
    found_ph = phosphopeptide_list[4]
    if len(found_ph) == 0:
         return False
    else:
        if n_ph == found_ph:
            return True
        else:
            return False

In [5]:
path = "Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx" #"Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx"
log2FC_threshold = 1 # Will keep sites with a log2FC grater than the threshold found at least in one of the time points
n_replicates = 2 # Will remove all site with the same or less number of replicates
all_phosphosites_found = True # If this is activated will keep only the sites where we know where are the phosphorylations happening
to_save = path.replace(".xlsx", f"_log2FC{log2FC_threshold}_nRep{n_replicates}_allPhosphoFound_{str(all_phosphosites_found)}.xlsx")

df = pd.read_excel(path)

In [6]:
df

,protein_ID,prot_name,site,EGF_full,EGF_starve,EGF_1,EGF_2,EGF_5,EGF_10,EGF_90,...,sd_INS5,sd_INS10,sd_INS90,sd_full_EGFnINS,sd_starve_EGFnINS,sd_EGFnINS1,sd_EGFnINS2,sd_EGFnINS5,sd_EGFnINS10,sd_EGFnINS90
0,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_1_S154~SGsGNFGGGR,2.750617,0,-0.075632,0.051413,0.553841,1.071310,0.614675,...,0.132813,0.082451,0.158647,0.240242,0.073607,0.213038,0.017743,0.137722,0.158750,0.203929
1,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_0~NQGGYGGSSSSSSYGSGR,-0.475987,0,-0.055121,-0.172318,-0.206575,-0.181191,-0.458281,...,0.122319,0.050334,0.136001,0.110507,0.109621,0.049720,0.281925,0.100414,0.150921,0.137177
2,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S266~NQGGYGGSSSsSSYGSGR,-0.726273,0,-0.103509,-0.389250,-0.566986,-0.520466,-0.750087,...,0.096032,0.092438,0.135065,0.190606,0.107901,0.087274,0.362086,0.085434,0.223696,0.084909
3,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_260_271_1_1_S264~NQGGYGGSsSSSSYGSGR,-0.337808,0,0.099901,0.026794,-0.030970,-0.071683,-0.061284,...,0.229781,0.228683,0.101860,0.015341,0.096143,0.177090,0.027352,0.391594,0.279341,0.065133
4,A0A2R8Y4L2,HNRNPA1L3,A0A2R8Y4L2_152_154_1_0~SGSGNFGGGR,1.341787,0,-0.045784,-0.057065,-0.181892,0.296514,0.152992,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34997,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_322_330_1_0~NSPQSSPTSTPK,0.385391,0,0.015958,0.080110,-0.013490,0.130227,0.036401,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34998,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_322_330_2_2_S322S325~NsPQsSPTSTPK,-0.254850,0,0.141716,-0.081345,-0.339977,-0.489481,-0.374868,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34999,Q9Y6Y0,IVNS1ABP,Q9Y6Y0_336_356_1_0~SLSFEMQQDELIEKPMSPMQYAR,-0.325955,0,-0.094084,-0.149544,-0.103233,-0.104764,-0.135981,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35000,Q9Y6Y8,SEC23IP,Q9Y6Y8_728_728_1_1_S728~GQEQsAQK,-0.032391,0,-0.000013,-0.117576,0.133564,0.161375,0.216911,...,0.581465,0.087426,0.298340,0.339735,0.015411,0.063062,0.138668,0.194455,0.166717,0.163315


In [7]:

for row in df.itertuples():
    df_index = row[0] 
    # Drop by log2FC threshold
    minimun = check_min_amplitude(row)
    maximun = check_max_amplitude(row)
    # print(type(minimun), minimun,type(minimun), maximun)
    if maximun >= log2FC_threshold or -1*minimun >= log2FC_threshold:
        pass
    else:
        df.drop(index=df_index, inplace=True)
        continue       
    # Drop by replicates
    replicates = int(row[25])
    if replicates <= n_replicates:
        df.drop(index=df_index, inplace=True)
        continue   
    # Drop sites that don't have all phosphorylation sites localized
    if all_phosphosites_found == True:
        site = row[3]
        all_localized = check_all_localized(site)
        if all_localized == False:
            df.drop(index=df_index, inplace=True)
            continue     
# Saving file
df.to_excel(to_save, index=False)

## Expressing the curve behaviour as qualitative change

In [66]:
path = "" #"Experiment/1_hTERT_HME1/Data/Processed/full_starve_continuous_FC_nrep_SD.xlsx"
error_allowed = 0.1 # Percentage error allowed to consider the step as "stay"
to_save = path.replace(".xlsx", "_qualitative_description.xlsx")

df = pd.read_excel(path)
# Adding colums for each of the conditions behaviour
df["EGF_qualitative"] = False
df["INS_qualitative"] = False
df["EGFnINS_qualitative"] = False
qualitative_list_names = ["EGF_qualitative", "INS_qualitative", "EGFnINS_qualitative"]
for row in df.itertuples():
    c = 4 # Time point data start in colum 4
    replicates = row[24] # number of replicates the phosphosite has
    qualitative_list = [] 
    string = ""
    #Check the maximum amplitude and calculate the error allowed based on that
    minimun = check_min_amplitude(row)
    maximun = check_max_amplitude(row)
    if minimun < 0: # if minimum is negative, transform it to positive
        minimun = minimun * -1    
    value_error_allowed = (max(maximun, minimun)*error_allowed)
    #Qualitative analysis (There are 6 possible steps)
    for i in range(21):
        # Skipping the transition from one condition to the other (from EGF_90 to full_INS, and to INS_90 to full_EGFnINS)
        if c+i in [10,17,24]: 
            qualitative_list.append(string[:-1]) #remove the last "_"
            string = ""
            continue
            # If I wanted to skip more steps I could include these values for the skipping part of the algorithm
            # if c+i in [4, 10, 9,         #full to cero, 90 to next full, 10 to 90
            #           11, 17, 16,       #full to cero, 90 to next full, 10 to 90
            #           18, 24, 23]:   #full to cero, 90 to next full, 10 to 90
        # Qualitative step        
        if (row[c+i] < row[c+i+1]-value_error_allowed):
            string = string+"up_"
        elif (row[c+i] > row[c+i+1]+value_error_allowed):
            string = string+"down_"
        else:
            string = string+"stay_"
    # Add the information to the dataframe
    c = 0 # This is for the tree conditions: EGF, INS and EGFnINS
    for element in qualitative_list_names:
        df.at[row[0], element] = qualitative_list[c]
        c += 1
# Saving file
df.to_excel(to_save, index=False)

FileNotFoundError: [Errno 2] No such file or directory: ''

here i am missing 
- from the Data_filtration file
    - the checking string networks and transcription factors
- from the Ochoa_filtering 
    - filter based on ochoa functional score
- from data_extractino
    - analyse information from the database

    